# Homework 2: BERT on AWS

In this homework, we will apply the BERT algorithm on Amazon Web Services.

This DataRec repository contains a pointer to accessing recommendation system data, installable via

`pip install datarec-lib`

### Task 1: Download the MovieLens 1m dataset. You should output a copy of the dataset on an AWS S3 bucket.

First, I set up aws CLI and ran `aws configure sso` with the Northwestern start URL and `us-east-2`.
aws s3 ls --profile mse-tl-dataeng300-EMR-549787090008

In [1]:
# aws sso login --profile mse-tl-dataeng300-EMR-549787090008
import os
import boto3
import pandas as pd

os.environ["AWS_SHARED_CREDENTIALS_FILE"] = "./.aws/credentials"
session = boto3.Session(
    profile_name="mse-tl-dataeng300-EMR-549787090008",
    region_name="us-east-2"
    )


In [2]:
s3 = session.client("s3")
response = s3.list_buckets()
if response['ResponseMetadata']['HTTPStatusCode'] == 200:
    print(response['Buckets'])


[{'Name': 'acharya-cui-sokolenko-mwaa', 'CreationDate': datetime.datetime(2026, 5, 1, 11, 48, 44, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::acharya-cui-sokolenko-mwaa'}, {'Name': 'acharya-de300-wi26', 'CreationDate': datetime.datetime(2026, 4, 29, 6, 17, 22, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::acharya-de300-wi26'}, {'Name': 'adams-agrawal-evensen-mwaa', 'CreationDate': datetime.datetime(2026, 4, 30, 23, 48, 34, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::adams-agrawal-evensen-mwaa'}, {'Name': 'ads-de300winter-airflow', 'CreationDate': datetime.datetime(2026, 4, 24, 22, 0, 35, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::ads-de300winter-airflow'}, {'Name': 'airflow-group6', 'CreationDate': datetime.datetime(2026, 4, 28, 22, 35, 13, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::airflow-group6'}, {'Name': 'airflowgroup5', 'CreationDate': datetime.datetime(2026, 5, 1, 7, 35, 34, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::airflowgroup5'}, {'Name': 'amde300', 'CreationDate': datetime

In [3]:
import os
import zipfile
import requests
import botocore

url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
bucket_name = "bhh-26spring"
keys = ["movies.dat", "ratings.dat", "users.dat"]

zip_path = "ml-1m.zip"
extract_dir = ""


def file_exists_in_s3(s3, bucket, key):
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except botocore.exceptions.ClientError as e:
        if e.response["Error"]["Code"] == "404":
            return False
        raise


def download_zip(url, zip_path):
    response = requests.get(url)
    response.raise_for_status()

    with open(zip_path, "wb") as f:
        f.write(response.content)


def unzip_file(zip_path, extract_dir):
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)


# Download and unzip locally
if not os.path.exists(zip_path):
    print("downloading the zip file...")
    download_zip(url, zip_path)
else:
    print("Zip file exists on local already, skipping")

if not os.path.exists(os.path.join(extract_dir, "ml-1m")):
    print("unzipping file...")
    unzip_file(zip_path, extract_dir)
else:
    print("files alr unzipped locally, skipping.")

# Upload missing files
for key in keys:
    s3_key = key
    local_path = os.path.join(extract_dir, "ml-1m", key)

    if file_exists_in_s3(s3, bucket_name, s3_key):
        print(f"{s3_key} already exists in S3, skipping.")
    else:
        print(f"Uploading {s3_key}...")
        s3.upload_file(local_path, bucket_name, s3_key)
        print(f"Uploaded {s3_key} to S3!")

Zip file exists on local already, skipping
files alr unzipped locally, skipping.
movies.dat already exists in S3, skipping.
ratings.dat already exists in S3, skipping.
users.dat already exists in S3, skipping.


### Task 2. Using only movies that are released before the year 1980, inclusive. Create embeddings for the BERT algorithm. For the same set of items (movies) in the dataset, you should create the embeddings once and output a copy of the necessary intermediate results on the S3 bucket.

In [4]:
movies_path = "ml-1m/movies.dat"
ratings_path = "ml-1m/ratings.dat"
users_path = "ml-1m/users.dat"
movies = pd.read_csv(
    movies_path,
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["movie_id", "title", "genres"]
)

ratings = pd.read_csv(
    ratings_path,
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

users = pd.read_csv(
    users_path,
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["user_id", "gender", "age", "occupation", "zip"]
)

In [5]:
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss
import re

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"  # for illustration, 66M model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
def extract_year(title):
    match = re.search(r"\((\d{4})\)", title)
    return int(match.group(1)) if match else None


movies["year"] = movies["title"].apply(extract_year)

items = movies[
    movies["year"].notna() & (movies["year"] <= 1980)
].copy()

items["text"] = items["title"] + ". Genres: " + items["genres"].str.replace("|", ", ")

print(f"Movies selected: {len(items)}")


Movies selected: 887


In [ ]:
@torch.no_grad()
def bert_embed(texts, max_len=128, batch_size=32):
    all_embs = []

    for i in range(0, len(texts), batch_size): # batching loop
        chunk = texts[i:i+batch_size]

        batch = tokenizer(
            chunk,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        out = encoder(**batch)
        cls = out.last_hidden_state[:, 0]
        emb = F.normalize(cls, dim=-1)

        all_embs.append(emb.cpu().numpy().astype("float32"))

        del batch, out, cls, emb
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    return np.vstack(all_embs)


def embed_to_vec(items, batch_size=32):
    texts = items["text"].tolist()
    item_id_list = items["movie_id"].tolist()

    item_vecs = bert_embed(texts, batch_size=batch_size)

    index = faiss.IndexFlatIP(item_vecs.shape[1])
    index.add(item_vecs)

    return item_vecs, item_id_list, index


item_vecs, item_id_list, index = embed_to_vec(items)

In [8]:
output_dir = "offline_bert_outputs"
os.makedirs(output_dir, exist_ok=True)
metadata_path = os.path.join(output_dir, "movies_pre_1980_metadata.csv")
vecs_path = os.path.join(output_dir, "movies_pre_1980_bert_vectors.npy")
ids_path = os.path.join(output_dir, "movies_pre_1980_movie_ids.npy")
faiss_path = os.path.join(output_dir, "movies_pre_1980_faiss.index")

items[["movie_id", "title", "genres", "year", "text"]].to_csv(metadata_path, index=False)
# print(items[["movie_id", "title", "genres", "year", "text"]])
np.save(vecs_path, item_vecs)
np.save(ids_path, np.array(item_id_list))
faiss.write_index(index, faiss_path)

In [10]:
def upload_to_s3(local_path, bucket_name, s3_key):
    s3.upload_file(local_path, bucket_name, s3_key)
    print(f"Uploaded s3://{bucket_name}/{s3_key}")

s3_prefix = "movielens/offline_bert_outputs"
for local_path in [metadata_path, vecs_path, ids_path, faiss_path]:
    filename = os.path.basename(local_path)
    upload_to_s3(
        local_path,
        bucket_name,
        f"{s3_prefix}/{filename}"
    )

Uploaded s3://bhh-26spring/movielens/offline_bert_outputs/movies_pre_1980_metadata.csv
Uploaded s3://bhh-26spring/movielens/offline_bert_outputs/movies_pre_1980_bert_vectors.npy
Uploaded s3://bhh-26spring/movielens/offline_bert_outputs/movies_pre_1980_movie_ids.npy
Uploaded s3://bhh-26spring/movielens/offline_bert_outputs/movies_pre_1980_faiss.index


### 3. Recommend five movies for each of the following users. The recommendations should be saved in a file on the S3 bucket containing User_Type, Last_Interaction_Time, other user summaries in the dataset,and a list of recommended movies:

Cold user: a user that the system has no data on.
Top user: a random user who has frequently rated movies (number of interactions among the top 5% of users).

In [11]:
# helper mappings
id_to_idx = {mid: i for i, mid in enumerate(item_id_list)}
id_to_title = dict(zip(items["movie_id"], items["title"]))

In [12]:
def build_user_text(user_id, ratings_df, items_df, N=3):
    hist = (
        ratings_df[ratings_df["user_id"] == user_id]
        .sort_values("timestamp")
        .tail(N)["movie_id"]
        .tolist()
    )

    # only keep movies that exist in the candidate item set
    valid_ids = set(items_df["movie_id"])
    hist = [mid for mid in hist if mid in valid_ids]

    if not hist:
        return "no history", set()

    text = (
        items_df.set_index("movie_id")
        .loc[hist, "text"]
        .tolist()
    )

    return " ".join(text), set(hist)

In [13]:
def recommend(user_id, ratings_df, items_df, item_id_list, index, k=5, N=3):
    user_text, seen = build_user_text(user_id, ratings_df, items_df, N=N)
    u = bert_embed([user_text])
    search_k = min(k + len(seen) + 20, len(item_id_list))
    scores, idx = index.search(u, search_k)
    recs = []
    for score, j in zip(scores[0], idx[0]):
        if j == -1:
            continue

        mid = item_id_list[j]

        if mid in seen:
            continue

        row = items_df[items_df["movie_id"] == mid].iloc[0]

        recs.append({
            "movie_id": int(mid),
            "title": row["title"],
            "genres": row["genres"],
            "year": int(row["year"]) if "year" in row and pd.notna(row["year"]) else None,
            "score": float(score)
        })

        if len(recs) == k:
            break

    return recs


In [14]:
# top user
user_counts = ratings.groupby("user_id").size().reset_index(name="num_interactions")
threshold = user_counts["num_interactions"].quantile(0.95)

top_users = user_counts[user_counts["num_interactions"] >= threshold]
top_user_id = top_users.sample(1, random_state=42)["user_id"].iloc[0]

top_pre_recs = recommend(
    user_id=top_user_id,
    ratings_df=ratings,
    items_df=items,
    item_id_list=item_id_list,
    index=index,
    k=5,
    N=3
)

In [15]:
# cold user
ratings_pre = ratings[ratings["movie_id"].isin(item_id_list)]

movie_stats = (
    ratings_pre
    .groupby("movie_id")
    .agg(
        rating_count=("rating", "count"),
        avg_rating=("rating", "mean")
    )
    .reset_index()
)

movie_stats["score"] = movie_stats["avg_rating"] * np.log1p(movie_stats["rating_count"])

cold_pre_recs = (
    movie_stats
    .sort_values("score", ascending=False)
    .head(5)
    .merge(items[["movie_id", "title", "genres", "year"]], on="movie_id")
    .to_dict(orient="records")
)

In [16]:
top_user_events = ratings[ratings["user_id"] == top_user_id]
top_user_info = users[users["user_id"] == top_user_id].iloc[0]

last_time_pre = pd.to_datetime(
    top_user_events["timestamp"].max(),
    unit="s"
).isoformat()

pre_1980_results = [
    {
        "Dataset": "pre_1980",
        "User_Type": "Cold user",
        "Last_Interaction_Time": None,
        "User_Summary": "No interaction history available. Recommendations use popular highly-rated pre-1980 movies.",
        "Recommended_Movies": cold_pre_recs
    },
    {
        "Dataset": "pre_1980",
        "User_Type": "Top user",
        "Last_Interaction_Time": last_time_pre,
        "User_Summary": {
            "user_id": int(top_user_id),
            "num_interactions": int(len(top_user_events)),
            "avg_rating": float(top_user_events["rating"].mean()),
            "gender": top_user_info["gender"],
            "age": int(top_user_info["age"]),
            "occupation": int(top_user_info["occupation"]),
            "zip": str(top_user_info["zip"])
        },
        "Recommended_Movies": top_pre_recs
    }
]

In [17]:
user_ratings = ratings[ratings["user_id"] == top_user_id]
user_pre = user_ratings[user_ratings["movie_id"].isin(item_id_list)]

liked = user_pre[user_pre["rating"] >= 4]

if len(liked) == 0:
    liked = user_pre  # fallback

liked_vecs = np.array([
    item_vecs[id_to_idx[mid]]
    for mid in liked["movie_id"]
    if mid in id_to_idx
])

user_vec = liked_vecs.mean(axis=0)
user_vec = user_vec / np.linalg.norm(user_vec)

In [18]:
cold = pre_1980_results[0]

print("User Type:", cold["User_Type"])
print("Last Interaction:", cold["Last_Interaction_Time"])
print("Summary:", cold["User_Summary"])

User Type: Cold user
Last Interaction: None
Summary: No interaction history available. Recommendations use popular highly-rated pre-1980 movies.


In [18]:
top = pre_1980_results[1]

print("User Type:", top["User_Type"])
print("Last Interaction:", top["Last_Interaction_Time"])
print("Summary:")
for k, v in top["User_Summary"].items():
    print(f"  {k}: {v}")

User Type: Top user
Last Interaction: 2003-02-15T22:38:13
Summary:
  user_id: 3519
  num_interactions: 666
  avg_rating: 3.045045045045045
  gender: F
  age: 25
  occupation: 14
  zip: 11215


In [19]:
top_user_events = ratings[ratings["user_id"] == top_user_id]

top_user_history = top_user_events.merge(
    movies[["movie_id", "title", "genres"]],
    on="movie_id",
    how="left"
)

top_user_history.head()

,user_id,movie_id,rating,timestamp,title,genres
0,3519,2987,3,968340999,Who Framed Roger Rabbit? (1988),Adventure|Animation|Film-Noir
1,3519,3791,3,969996802,Footloose (1984),Drama
2,3519,3793,4,976889941,X-Men (2000),Action|Sci-Fi
3,3519,2991,3,971201646,Live and Let Die (1973),Action
4,3519,2054,1,968340601,"Honey, I Shrunk the Kids (1989)",Adventure|Children's|Comedy|Fantasy|Sci-Fi


In [20]:
print("Cold user recs:")
for m in pre_1980_results[0]["Recommended_Movies"]:
    print(m["title"])

print("\nTop user recs:")
for m in pre_1980_results[1]["Recommended_Movies"]:
    print(m["title"])

Cold user recs:
Star Wars: Episode IV - A New Hope (1977)
Godfather, The (1972)
Star Wars: Episode V - The Empire Strikes Back (1980)
Casablanca (1942)
One Flew Over the Cuckoo's Nest (1975)

Top user recs:
All the King's Men (1949)
Long Goodbye, The (1973)
To Sir with Love (1967)
Candidate, The (1972)
Buddy Holly Story, The (1978)


### 4. Repeat steps 2 and 3 but with the full set of data. You should be able to reuse your work from earlier.

In [21]:
full_items = movies.copy()

full_items["text"] = (
    full_items["title"]
    + ". Genres: "
    + full_items["genres"].str.replace("|", ", ", regex=False)
)

full_item_vecs = bert_embed(full_items["text"].tolist())

full_item_id_list = full_items["movie_id"].tolist()

full_index = faiss.IndexFlatIP(full_item_vecs.shape[1])
full_index.add(full_item_vecs)

### 5. Choose and rate 10 movies and create a “user profile” for yourself. Save your user profile on the S3 bucket. Recommend 5 movies for yourself and save the results on the S3 bucket.

In [22]:
# check if not taken user id
users[users["user_id"] == 999999]

,user_id,gender,age,occupation,zip


In [23]:
def movie_lookup(text):
    print(movies[movies["title"].str.contains(text, case=False, na=False)])

searches = ["Shawshank Redemption", "Toy Story", "Fight Club", "Star Wars", "Avengers"]
for search in searches:
    print("\nSearch for:", search)
    movie_lookup(search)


Search for: Shawshank Redemption
     movie_id                             title genres  year
315       318  Shawshank Redemption, The (1994)  Drama  1994

Search for: Toy Story
      movie_id               title                       genres  year
0            1    Toy Story (1995)  Animation|Children's|Comedy  1995
3045      3114  Toy Story 2 (1999)  Animation|Children's|Comedy  1999

Search for: Fight Club
      movie_id              title genres  year
2890      2959  Fight Club (1999)  Drama  1999

Search for: Star Wars
      movie_id                                              title  \
257        260          Star Wars: Episode IV - A New Hope (1977)   
1178      1196  Star Wars: Episode V - The Empire Strikes Back...   
1192      1210  Star Wars: Episode VI - Return of the Jedi (1983)   
2559      2628   Star Wars: Episode I - The Phantom Menace (1999)   

                                   genres  year  
257       Action|Adventure|Fantasy|Sci-Fi  1977  
1178    Action|Adventure

In [24]:
my_user_id = 999999

my_ratings = pd.DataFrame([
    {"user_id": my_user_id, "movie_id": 1, "rating": 5, "timestamp": pd.Timestamp.now().timestamp()},     # Toy Story
    {"user_id": my_user_id, "movie_id": 260, "rating": 5, "timestamp": pd.Timestamp.now().timestamp()},   # Star Wars
    {"user_id": my_user_id, "movie_id": 1196, "rating": 5, "timestamp": pd.Timestamp.now().timestamp()},  # Empire Strikes Back
    {"user_id": my_user_id, "movie_id": 318, "rating": 5, "timestamp": pd.Timestamp.now().timestamp()},   # Shawshank Redemption
    {"user_id": my_user_id, "movie_id": 1270, "rating": 4, "timestamp": pd.Timestamp.now().timestamp()},  # Back to the Future
    {"user_id": my_user_id, "movie_id": 1210, "rating": 4, "timestamp": pd.Timestamp.now().timestamp()},  # Return of the Jedi
    {"user_id": my_user_id, "movie_id": 2571, "rating": 5, "timestamp": pd.Timestamp.now().timestamp()},  # Matrix
    {"user_id": my_user_id, "movie_id": 2959, "rating": 4, "timestamp": pd.Timestamp.now().timestamp()},  # Fight Club
    {"user_id": my_user_id, "movie_id": 2153, "rating": 5, "timestamp": pd.Timestamp.now().timestamp()},  # The Avengers
    {"user_id": my_user_id, "movie_id": 3114, "rating": 4, "timestamp": pd.Timestamp.now().timestamp()},  # Toy Story 2
])

In [25]:
ratings_with_me = pd.concat([ratings, my_ratings], ignore_index=True)

my_recs = recommend(
    user_id=my_user_id,
    ratings_df=ratings_with_me,
    items_df=full_items,
    item_id_list=full_item_id_list,
    index=full_index,
    k=5,
    N=10
)

my_recs

[{'movie_id': 2628,
  'title': 'Star Wars: Episode I - The Phantom Menace (1999)',
  'genres': 'Action|Adventure|Fantasy|Sci-Fi',
  'year': 1999,
  'score': 0.9702330231666565},
 {'movie_id': 1544,
  'title': 'Lost World: Jurassic Park, The (1997)',
  'genres': 'Action|Adventure|Sci-Fi|Thriller',
  'year': 1997,
  'score': 0.9688523411750793},
 {'movie_id': 329,
  'title': 'Star Trek: Generations (1994)',
  'genres': 'Action|Adventure|Sci-Fi',
  'year': 1994,
  'score': 0.9676699042320251},
 {'movie_id': 1205,
  'title': 'Transformers: The Movie, The (1986)',
  'genres': "Action|Animation|Children's|Sci-Fi|Thriller|War",
  'year': 1986,
  'score': 0.967428982257843},
 {'movie_id': 1371,
  'title': 'Star Trek: The Motion Picture (1979)',
  'genres': 'Action|Adventure|Sci-Fi',
  'year': 1979,
  'score': 0.9665452241897583}]

In [26]:
my_profile = my_ratings.merge(
    movies[["movie_id", "title", "genres"]],
    on="movie_id",
    how="left"
)

my_profile_path = "my_user_profile.csv"
my_profile.to_csv(my_profile_path, index=False)

s3.upload_file(
    my_profile_path,
    bucket_name,
    "movielens/my_user_profile/my_user_profile.csv"
)

print("uploaded my user profile to S3")

uploaded my user profile to S3


In [27]:
my_results = {
    "User_Type": "Self user profile",
    "User_ID": my_user_id,
    "Last_Interaction_Time": pd.to_datetime(
        my_ratings["timestamp"].max(),
        unit="s"
    ).isoformat(),
    "User_Summary": {
        "num_interactions": len(my_ratings),
        "avg_rating": float(my_ratings["rating"].mean()),
        "profile_description": "Self-created user profile based on 10 manually rated movies."
    },
    "Rated_Movies": my_profile.to_dict(orient="records"),
    "Recommended_Movies": my_recs
}

In [28]:
import json
my_results_path = "my_recommendations.json"

with open(my_results_path, "w") as f:
    json.dump(my_results, f, indent=2)

s3.upload_file(
    my_results_path,
    bucket_name,
    "movielens/my_user_profile/my_recommendations.json"
)

print("uploaded my recommendations to S3")

uploaded my recommendations to S3
